In [1]:
import ollama
import numpy as np
import pandas as pd

# Function to get embeddings from Ollama
def get_ollama_embedding(text, model="llama3.2"):
    """Get embeddings from Ollama for a single text"""
    response = ollama.embeddings(model=model, prompt=text)
    return response['embedding']

# Function to batch process documents
def batch_get_embeddings(documents, batch_size=10, model="llama3.2"):
    """Process documents in batches"""
    all_embeddings = []
    for i in range(0, len(documents), batch_size):
        batch = documents[i:i+batch_size]
        batch_embeddings = [get_ollama_embedding(doc, model) for doc in batch]
        all_embeddings.extend(batch_embeddings)
        print(f"Processed {i+len(batch)}/{len(documents)} documents")
    return np.array(all_embeddings)

In [4]:
from analysis.models.data import Data

with open("../data/data.json", "r") as f:
    data = Data.model_validate_json(f.read())

system = data.systems["20250203_openhands_4x_scaled"]

rows = []
for instance in data.dataset.instances:
    row = {
        "instance_id": instance.instance_id,
        "problem_statement": instance.problem_statement,
        "resolved": system.results.is_resolved(instance.instance_id),
    }
    rows.append(row)

df = pd.DataFrame(rows)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df['problem_statement'], df['resolved'], test_size=0.2, random_state=42
)

# Generate embeddings using Ollama
X_train_embeddings = batch_get_embeddings(X_train.tolist())
X_test_embeddings = batch_get_embeddings(X_test.tolist())

# Continue with model training
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_embeddings)
X_test_scaled = scaler.transform(X_test_embeddings)

Processed 10/400 documents
Processed 20/400 documents
Processed 30/400 documents
Processed 40/400 documents
Processed 50/400 documents
Processed 60/400 documents
Processed 70/400 documents
Processed 80/400 documents
Processed 90/400 documents
Processed 100/400 documents
Processed 110/400 documents
Processed 120/400 documents
Processed 130/400 documents
Processed 140/400 documents
Processed 150/400 documents
Processed 160/400 documents
Processed 170/400 documents
Processed 180/400 documents
Processed 190/400 documents
Processed 200/400 documents
Processed 210/400 documents
Processed 220/400 documents
Processed 230/400 documents
Processed 240/400 documents
Processed 250/400 documents
Processed 260/400 documents
Processed 270/400 documents
Processed 280/400 documents
Processed 290/400 documents
Processed 300/400 documents
Processed 310/400 documents
Processed 320/400 documents
Processed 330/400 documents
Processed 340/400 documents
Processed 350/400 documents
Processed 360/400 documents
P

LogisticRegression(max_iter=1000)

In [20]:
# Compute ROC curve against a particular system in the data
from sklearn.metrics import roc_curve
import altair as alt

def roc(y_scores: pd.Series, y_true: pd.Series) -> alt.Chart:
    """
    Compute the ROC curve for a given system.
    """
    df = pd.DataFrame({
        "y_scores": y_scores,
        "y_true": y_true,
    })

    fpr, tpr, thresholds = roc_curve(df['y_true'], df['y_scores'])
    curve = pd.DataFrame({
        "fpr": fpr,
        "tpr": tpr,
        "threshold": thresholds
    })

    p_ratio = df['y_true'].sum() / len(df)
    curve["accuracy"] = p_ratio * curve["tpr"] + (1 - p_ratio) * (1 - curve["fpr"])

    chart = alt.Chart(curve).mark_line().encode(
        x=alt.X("fpr", title="False Positive Rate"),
        y=alt.Y("tpr", title="True Positive Rate"),
        tooltip=["fpr", "tpr", "threshold", "accuracy"],
    )

    # Compute the threshold/accuracy pair with the highest accuracy
    best_threshold = curve.loc[curve['accuracy'].idxmax(), 'threshold']
    best_accuracy = curve['accuracy'].max()
    print(f"Best threshold: {best_threshold:.2f}, Best accuracy: {best_accuracy:.2f}")

    return chart

In [21]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

roc(
    pd.Series(model.predict_proba(X_test_scaled)[:, 1], index=X_test.index),
    pd.Series(y_test, index=X_test.index),
)

Best threshold: 0.01, Best accuracy: 0.60


alt.Chart(...)

In [22]:
from sklearn.svm import SVR

model = SVR(kernel='rbf')
model.fit(X_train_scaled, y_train)

roc(
    pd.Series(model.predict(X_test_scaled), index=X_test.index),
    pd.Series(y_test, index=X_test.index),
)

Best threshold: 0.50, Best accuracy: 0.61


alt.Chart(...)

In [24]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train_scaled, y_train)

roc(
    pd.Series(model.predict_proba(X_test_scaled)[:, 1], index=X_test.index),
    pd.Series(y_test, index=X_test.index),
)

Best threshold: 0.67, Best accuracy: 0.61


alt.Chart(...)

In [25]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train_scaled, y_train)

roc(
    pd.Series(model.predict_proba(X_test_scaled)[:, 1], index=X_test.index),
    pd.Series(y_test, index=X_test.index),
)


Best threshold: 0.51, Best accuracy: 0.63


alt.Chart(...)

In [27]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier()
model.fit(X_train_scaled, y_train)

roc(
    pd.Series(model.predict_proba(X_test_scaled)[:, 1], index=X_test.index),
    pd.Series(y_test, index=X_test.index),
)


Best threshold: 0.40, Best accuracy: 0.65


alt.Chart(...)

In [28]:
from sklearn.naive_bayes import GaussianNB

model = GaussianNB()
model.fit(X_train_scaled, y_train)

roc(
    pd.Series(model.predict_proba(X_test_scaled)[:, 1], index=X_test.index),
    pd.Series(y_test, index=X_test.index),
)

Best threshold: 0.00, Best accuracy: 0.62


alt.Chart(...)